In [ ]:
# Establish directory structure and write production-grade RAG FastAPI application
import os

# Create necessary directories
os.makedirs("app", exist_ok=True)

# 1. WRITE REQUIREMENTS.TXT
requirements_content = """fastapi>=0.110.0
uvicorn>=0.28.0
pydantic>=2.6.0
pydantic-settings>=2.2.0
langchain>=0.1.12
langchain-community>=0.0.28
langchain-openai>=0.0.9
chromadb>=0.4.24
python-multipart>=0.0.9
pypdf>=4.1.0
tiktoken>=0.6.0
"""

with open("requirements.txt", "w") as f:
    f.write(requirements_content)

# 2. WRITE README.MD
readme_content = """# Production-Ready Modular RAG API

This is a highly modular, clean, and production-grade Retrieval-Augmented Generation (RAG) API built with FastAPI, Pydantic v2, LangChain, and ChromaDB.

## 🚀 Project Architecture
```
├── app/
│   ├── __init__.py
│   ├── main.py          # FastAPI application entrypoint & API routes
│   ├── config.py        # Environment variables & configuration management (Pydantic Settings)
│   ├── schemas.py       # Pydantic models for request/response payloads
│   ├── database.py      # Vector DB connection lifecycle and client setup
│   ├── ingestion.py     # Document text extraction, chunking, and embedding logic
│   └── pipeline.py      # Core RAG retrieval and LLM prompt orchestration
├── requirements.txt
└── README.md
```

## 🛠️ How to Setup & Run
1. Install dependencies:
   ```bash
   pip install -r requirements.txt
   ```
2. Set environment variables:
   ```bash
   export OPENAI_API_KEY="your-openai-key-here"
   ```
3. Run the development server:
   ```bash
   uvicorn app.main:app --reload
   ```
"""

with open("README.md", "w") as f:
    f.write(readme_content)

# 3. WRITE APP/__INIT__.PY
with open("app/__init__.py", "w") as f:
    f.write("# Initialize app package\n")

# 4. WRITE APP/CONFIG.PY
config_content = """from pydantic_settings import BaseSettings, SettingsConfigDict

class Settings(BaseSettings):
    PROJECT_NAME: str = "Production RAG API"
    VERSION: str = "1.0.0"

    # Vector DB settings
    CHROMA_PERSIST_DIR: str = "./chroma_db"
    CHROMA_COLLECTION_NAME: str = "rag_collection"

    # Model Settings (Using OpenAI as default; easily swappable via inheritance)
    OPENAI_API_KEY: str = ""
    EMBEDDING_MODEL: str = "text-embedding-3-small"
    LLM_MODEL: str = "gpt-4o-mini"
    LLM_TEMPERATURE: float = 0.0

    # Chunking settings
    CHUNK_SIZE: int = 500
    CHUNK_OVERLAP: int = 50

    model_config = SettingsConfigDict(env_file=".env", env_file_encoding="utf-8", extra="ignore")

settings = Settings()
"""

with open("app/config.py", "w") as f:
    f.write(config_content)

# 5. WRITE APP/SCHEMAS.PY
schemas_content = """from pydantic import BaseModel, Field
from typing import List, Dict, Any, Optional

# --- Ingestion Schemas ---
class IngestTextRequest(BaseModel):
    text: str = Field(..., description="Raw textual context to ingest.")
    metadata: Optional[Dict[str, Any]] = Field(default=None, description="Optional metadata associated with the text chunk.")

class IngestResponse(BaseModel):
    status: str
    message: str
    inserted_chunks: int

# --- Retrieval Schemas ---
class RetrieveRequest(BaseModel):
    query: str = Field(..., description="Semantic search query to retrieve context matching documents.")
    top_k: int = Field(default=4, description="Number of document chunks to retrieve.")

class RetrievedChunk(BaseModel):
    page_content: str
    metadata: Dict[str, Any]
    score: Optional[float] = None

class RetrieveResponse(BaseModel):
    query: str
    results: List[RetrievedChunk]

# --- Generation (Ask) Schemas ---
class AskRequest(BaseModel):
    query: str = Field(..., description="The prompt or question to ask the LLM backed by internal context.")
    top_k: int = Field(default=4, description="Number of dynamic reference contexts to retrieve.")

class AskResponse(BaseModel):
    query: str
    answer: str
    sources: List[RetrievedChunk]
"""

with open("app/schemas.py", "w") as f:
    f.write(schemas_content)

# 6. WRITE APP/DATABASE.PY (Escaping inner docstrings to avoid syntax errors)
database_content = """import os
from typing import Generator
from langchain_community.vectorstores import Chroma
from langchain_openai import OpenAIEmbeddings
from app.config import settings

class VectorDBManager:
    \"\"\"
    Manages the lifecycle, connection, and initialization of the Vector database.
    Uses a singleton-like pattern or connection pool approach for FastAPI Lifespan.
    \"\"\"
    def __init__(self):
        self._embeddings = None
        self._vector_store = None

    def get_embeddings(self) -> OpenAIEmbeddings:
        if not self._embeddings:
            # Highly modular - can swap this implementation with Ollama, HuggingFace, etc.
            self._embeddings = OpenAIEmbeddings(
                model=settings.EMBEDDING_MODEL,
                openai_api_key=settings.OPENAI_API_KEY
            )
        return self._embeddings

    def get_vector_store(self) -> Chroma:
        if not self._vector_store:
            embeddings = self.get_embeddings()
            self._vector_store = Chroma(
                collection_name=settings.CHROMA_COLLECTION_NAME,
                persist_directory=settings.CHROMA_PERSIST_DIR,
                embedding_function=embeddings
            )
        return self._vector_store

# Instantiate database manager
db_manager = VectorDBManager()
"""

with open("app/database.py", "w") as f:
    f.write(database_content)

# 7. WRITE APP/INGESTION.PY (Escaping inner docstrings to avoid syntax errors)
ingestion_content = """import io
from typing import List
from fastapi import UploadFile, HTTPException
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from app.config import settings
from app.database import db_manager
import pypdf

class DocumentIngester:
    def __init__(self):
        self.text_splitter = RecursiveCharacterTextSplitter(
            chunk_size=settings.CHUNK_SIZE,
            chunk_overlap=settings.CHUNK_OVERLAP
        )

    def extract_text_from_file(self, file: UploadFile) -> str:
        \"\"\"
        Extracts raw string content dynamically from TXT and PDF files.
        \"\"\"
        filename = file.filename.lower() if file.filename else ""
        try:
            contents = file.file.read()
            if filename.endswith(".pdf"):
                pdf_reader = pypdf.PdfReader(io.BytesIO(contents))
                text = ""
                for page in pdf_reader.pages:
                    page_text = page.extract_text()
                    if page_text:
                        text += page_text + "\\n"
                return text
            elif filename.endswith(".txt") or filename == "":
                return contents.decode("utf-8")
            else:
                raise HTTPException(
                    status_code=400,
                    detail="Unsupported file format. Please upload a .txt or .pdf file."
                )
        except Exception as e:
            raise HTTPException(
                status_code=500,
                detail=f"Failed to process uploaded file: {str(e)}"
            )

    async def ingest_raw_text(self, text: str, metadata: dict = None) -> int:
        \"\"\"
        Chunks raw string and adds documents to the vector store database.
        \"\"\"
        if not text.strip():
            return 0

        metadata = metadata or {}
        metadata["source"] = metadata.get("source", "raw_text_api")

        chunks = self.text_splitter.split_text(text)
        documents = [Document(page_content=chunk, metadata=metadata) for chunk in chunks]

        vector_store = db_manager.get_vector_store()
        vector_store.add_documents(documents)
        return len(documents)

ingester = DocumentIngester()
"""

with open("app/ingestion.py", "w") as f:
    f.write(ingestion_content)

# 8. WRITE APP/PIPELINE.PY (Escaping inner docstrings to avoid syntax errors)
pipeline_content = """from typing import List, Tuple, Dict, Any
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from app.config import settings
from app.database import db_manager
from app.schemas import RetrievedChunk

class RAGPipeline:
    def __init__(self):
        # Initialize model components
        self.llm = ChatOpenAI(
            model=settings.LLM_MODEL,
            temperature=settings.LLM_TEMPERATURE,
            openai_api_key=settings.OPENAI_API_KEY
        )
        self.output_parser = StrOutputParser()

        # Design Grounded Prompt Template avoiding LLM hallucination
        self.prompt_template = ChatPromptTemplate.from_messages([
            ("system",
             "You are an expert AI system assistant. You are strictly grounded on the provided source context.\\n\\n"
             "Instructions:\\n"
             "1. Rely ONLY on the clear facts explicitly mentioned in the context.\\n"
             "2. If the context does not contain sufficient facts to fully answer the query, state honestly: "
             "'I don't know based on the provided context.'\\n"
             "3. Do not invent or extrapolate outside the provided reference content.\\n\\n"
             "CONTEXT:\\n{context}"),
            ("human", "{query}")
        ])

    def retrieve_context(self, query: str, top_k: int = 4) -> List[Tuple[Any, float]]:
        \"\"\"
        Performs semantic similarity search directly against Vector DB.
        \"\"\"
        vector_store = db_manager.get_vector_store()
        # Returns tuples of (Document, score)
        results = vector_store.similarity_search_with_relevance_scores(query, k=top_k)
        return results

    async def generate_grounded_answer(self, query: str, top_k: int = 4) -> Tuple[str, List[RetrievedChunk]]:
        \"\"\"
        Performs end-to-end grounded query answering orchestration.
        \"\"\"
        retrieved_docs_with_scores = self.retrieve_context(query, top_k=top_k)

        # Format prompt variables
        context_blocks = []
        sources = []

        for doc, score in retrieved_docs_with_scores:
            context_blocks.append(doc.page_content)
            sources.append(RetrievedChunk(
                page_content=doc.page_content,
                metadata=doc.metadata,
                score=score
            ))

        formatted_context = "\\n\\n---\\n\\n".join(context_blocks)

        # Create langchain chain
        chain = self.prompt_template | self.llm | self.output_parser

        # Async prompt inference invocation
        answer = await chain.ainvoke({
            "context": formatted_context if formatted_context else "No context available.",
            "query": query
        })

        return answer, sources

rag_pipeline = RAGPipeline()
"""

with open("app/pipeline.py", "w") as f:
    f.write(pipeline_content)

# 9. WRITE APP/MAIN.PY (Escaping inner docstrings to avoid syntax errors)
main_content = """from fastapi import FastAPI, UploadFile, File, Form, HTTPException, status
from typing import List, Optional
from contextlib import asynccontextmanager
import uvicorn

from app.config import settings
from app.schemas import (
    IngestTextRequest, IngestResponse,
    RetrieveRequest, RetrieveResponse, RetrievedChunk,
    AskRequest, AskResponse
)
from app.ingestion import ingester
from app.pipeline import rag_pipeline

@asynccontextmanager
async def lifespan(app: FastAPI):
    # Startup Hook (Verify API Key or Setup Directory Lifecycle if needed)
    if not settings.OPENAI_API_KEY:
        import os
        # Fallback to check process env directly
        settings.OPENAI_API_KEY = os.getenv("OPENAI_API_KEY", "")
    yield
    # Shutdown Hook (Clean up vector DB connections if required)
    pass

app = FastAPI(
    title=settings.PROJECT_NAME,
    version=settings.VERSION,
    lifespan=lifespan
)

@app.get("/", tags=["General"])
async def root():
    return {
        "service": settings.PROJECT_NAME,
        "status": "healthy",
        "version": settings.VERSION
    }

# --- 1. POST /ingest (Text & Files Support) ---
@app.post("/ingest", response_model=IngestResponse, tags=["Ingestion"])
async def ingest_document(
    file: Optional[UploadFile] = File(None),
    raw_text: Optional[str] = Form(None),
    source_name: Optional[str] = Form(None)
):
    \"\"\"
    Upload PDF/TXT files or submit raw text to chunk, embed, and store.
    \"\"\"
    if not file and not raw_text:
        raise HTTPException(
            status_code=status.HTTP_400_BAD_REQUEST,
            detail="Provide either an uploaded file or raw_text parameter."
        )

    try:
        metadata = {"source": source_name or "direct_upload"}
        if file:
            metadata["filename"] = file.filename
            extracted_text = ingester.extract_text_from_file(file)
            inserted = await ingester.ingest_raw_text(extracted_text, metadata)
        else:
            inserted = await ingester.ingest_raw_text(raw_text, metadata)

        return IngestResponse(
            status="success",
            message=f"Successfully integrated documents into database system.",
            inserted_chunks=inserted
        )
    except HTTPException as http_exc:
        raise http_exc
    except Exception as e:
        raise HTTPException(
            status_code=status.HTTP_500_INTERNAL_SERVER_ERROR,
            detail=f"An unexpected ingestion exception occurred: {str(e)}"
        )

# --- 2. POST /retrieve (Debugging Similarity Context) ---
@app.post("/retrieve", response_model=RetrieveResponse, tags=["Search"])
async def retrieve_context(payload: RetrieveRequest):
    \"\"\"
    Submit queries to query exact context chunks matched by relevance vector scores.
    \"\"\"
    try:
        results = rag_pipeline.retrieve_context(payload.query, top_k=payload.top_k)
        retrieved_chunks = [
            RetrievedChunk(
                page_content=doc.page_content,
                metadata=doc.metadata,
                score=score
            ) for doc, score in results
        ]
        return RetrieveResponse(query=payload.query, results=retrieved_chunks)
    except Exception as e:
        raise HTTPException(
            status_code=status.HTTP_500_INTERNAL_SERVER_ERROR,
            detail=f"An internal search exception occurred: {str(e)}"
        )

# --- 3. POST /ask (RAG Endpoint) ---
@app.post("/ask", response_model=AskResponse, tags=["Inference"])
async def ask_question(payload: AskRequest):
    \"\"\"
    Answers client query leveraging structured contextual retrieval with strict fallback boundaries.
    \"\"\"
    if not settings.OPENAI_API_KEY:
        raise HTTPException(
            status_code=status.HTTP_503_SERVICE_UNAVAILABLE,
            detail="OpenAI API Key is not configured on the server settings."
        )
    try:
        answer, sources = await rag_pipeline.generate_grounded_answer(payload.query, top_k=payload.top_k)
        return AskResponse(query=payload.query, answer=answer, sources=sources)
    except Exception as e:
        raise HTTPException(
            status_code=status.HTTP_500_INTERNAL_SERVER_ERROR,
            detail=f"An issue occurred during prompt engineering pipeline execution: {str(e)}"
        )

if __name__ == "__main__":
    uvicorn.run("app.main:app", host="0.0.0.0", port=8000, reload=True)
"""

with open("app/main.py", "w") as f:
    f.write(main_content)

print("✔ Project files created successfully! All components have been generated.")


✔ Project files created successfully! All components have been generated.
